# Inferencia estadística

Hasta este capítulo el flujo era: **dados los parámetros**, calcular
probabilidades. Inferencia invierte el sentido: **dada una muestra**, qué
podemos afirmar sobre los parámetros que la generaron.

Tres herramientas centrales:

- **Intervalos de confianza (IC)** — rango plausible para $\mu$, $p$ o
  $\sigma^2$.
- **Pruebas de hipótesis** — decisión binaria con control de error tipo I.
- **Tamaño muestral** — cuántos datos hacen falta para una precisión dada.

Las preguntas que veníamos arrastrando aterrizan acá: cuál es el rango
plausible para la espera media de la clínica, qué intervalo cubre la verdadera
proporción de «sí» en la encuesta y si los datos permiten rechazar la afirmación
de la línea de producción sobre su tasa de defectos.


## Preguntas de inicio

1. La clínica midió 36 esperas con $\bar{x} = 12$ minutos. ¿Cuál es el rango
   plausible para la espera media real $\mu$?
2. La encuesta tomó $n = 400$ personas y 200 dijeron «sí». ¿Cuál es el
   intervalo de confianza para la verdadera proporción $p$?
3. La fábrica afirma que la espera media en su línea de inspección es 11
   minutos. Con la muestra observada, ¿podemos rechazar esa afirmación?
4. ¿Cuántas personas necesita la encuesta para cerrar el margen de error en
   ±3% al 95% de confianza?


## Importaciones

In [ ]:
import numpy as np
import pandas as pd

from core import Observations, Settings
from exercises import (
    IntervalContainsInput,
    NumericAnswerInput,
    verify_interval_contains,
    verify_numeric_answer,
)
from inference import (
    MeanKnownVarianceInput,
    MeanUnknownVarianceInput,
    OneSampleMeanTestInput,
    ProportionInput,
    SampleSizeForMeanInput,
    SampleSizeForProportionInput,
    VarianceInput,
    build_confidence_interval_for_mean_known_variance,
    build_confidence_interval_for_mean_unknown_variance,
    build_confidence_interval_for_proportion,
    build_confidence_interval_for_variance,
    sample_size_for_mean,
    sample_size_for_proportion,
    test_one_sample_mean,
)
from inference.hypothesis_tests import Alternative
from sampling import BootstrapInput, bootstrap_mean
from visualization import (
    BootstrapDistributionChartInput,
    chart_bootstrap_distribution,
)
from widgets import MeanCIExplorerInput, build_mean_ci_explorer

In [ ]:
settings = Settings()

## IC para la espera media con $\sigma$ conocido

Tenemos $n = 36$ esperas con $\bar{x} = 12$ y, por datos históricos,
$\sigma = 3$. Buscamos un IC al 95%.

**Paso 1 — pivot.** Por la ecuación (4.3) del Capítulo 4 sabemos que el
promedio estandarizado tiende a Normal estándar:

$$ Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \sim \mathcal{N}(0, 1) \tag{5.1} $$

**Paso 2 — intervalo.** Despejamos $\mu$ usando el cuantil
$z_{1 - \alpha/2}$ de la Normal estándar (capítulo 3, ecuación 3.5):

$$ \bar{x} \;\pm\; z_{1 - \alpha/2}\,\frac{\sigma}{\sqrt{n}} \tag{5.2} $$

**Paso 3 — números.** Con $z_{0{,}975} = 1{,}96$, $\sigma = 3$, $n = 36$:

$$ 12 \;\pm\; 1{,}96 \cdot \frac{3}{6} = 12 \;\pm\; 0{,}98 = (11{,}02,\ 12{,}98) $$


In [ ]:
clinic_known_input = MeanKnownVarianceInput(
    sample_mean=12.0,
    population_standard_deviation=3.0,
    sample_size=36,
    confidence_level=0.95,
)
clinic_known_interval = build_confidence_interval_for_mean_known_variance(clinic_known_input)
clinic_known_interval

### Qué quiere decir «95% de confianza»

**No** es «el verdadero $\mu$ está con probabilidad 0,95 en este intervalo».
El verdadero $\mu$ es una constante. Lo aleatorio es la muestra y, por lo
tanto, el intervalo. La afirmación correcta es:

> Si repitiéramos este procedimiento muchas veces, **el 95% de los
> intervalos generados** contendría a $\mu$.

El widget de abajo lo materializa: cada línea es un intervalo construido
sobre una muestra distinta.


In [ ]:
ci_explorer_input = MeanCIExplorerInput(settings=settings)
build_mean_ci_explorer(ci_explorer_input)

## IC con $\sigma$ desconocido — la $t$ de Student

En la práctica casi nunca conocemos $\sigma$. Lo estimamos con $s$ (ecuación
1.3 del Capítulo 1) y el pivote cambia: deja de ser Normal. **Paso 1.** Si
las observaciones son Normales, vale:

$$ T = \frac{\bar{X} - \mu}{s/\sqrt{n}} \sim t_{n-1} \tag{5.3} $$

**Paso 2.** Reemplazamos $z_{1 - \alpha/2}$ por $t_{n-1,\,1 - \alpha/2}$ en
(5.2). Para $n$ grande la $t$ converge a la Normal y los dos IC coinciden.


In [ ]:
clinic_unknown_input = MeanUnknownVarianceInput(
    sample_mean=12.0,
    sample_standard_deviation=3.0,
    sample_size=36,
    confidence_level=0.95,
)
clinic_unknown_interval = build_confidence_interval_for_mean_unknown_variance(clinic_unknown_input)
clinic_unknown_interval

## IC para una proporción

En la encuesta, 200 de 400 dicen «sí». La proporción observada es
$\hat{p} = 0{,}5$. **Paso 1:** la Binomial subyacente (ecuación 3.1) tiene
varianza $np(1-p)$. **Paso 2:** por el TCL (ecuación 4.4),
$\hat{p}$ es aproximadamente Normal. **Paso 3:** IC de Wald al $1 - \alpha$:

$$ \hat{p} \;\pm\; z_{1 - \alpha/2}\,\sqrt{\frac{\hat{p}(1 - \hat{p})}{n}} \tag{5.4} $$


In [ ]:
poll_input = ProportionInput(
    successes=200,
    sample_size=400,
    confidence_level=0.95,
)
poll_interval = build_confidence_interval_for_proportion(poll_input)
poll_interval

## IC para la varianza — la $\chi^2$

Si las $X_i$ son Normales con varianza $\sigma^2$, entonces:

$$ \frac{(n-1)\,S^2}{\sigma^2} \sim \chi^2_{n-1} \tag{5.5} $$

Despejando $\sigma^2$ con los cuantiles $\chi^2_{n-1,\,\alpha/2}$ y
$\chi^2_{n-1,\,1 - \alpha/2}$:

$$ \left(\frac{(n-1)\,s^2}{\chi^2_{n-1,\,1 - \alpha/2}},\ \frac{(n-1)\,s^2}{\chi^2_{n-1,\,\alpha/2}}\right) \tag{5.6} $$

Aplicado a la clínica con $s^2 = 9$ y $n = 36$:


In [ ]:
clinic_variance_input = VarianceInput(
    sample_variance=9.0,
    sample_size=36,
    confidence_level=0.95,
)
clinic_variance_interval = build_confidence_interval_for_variance(clinic_variance_input)
clinic_variance_interval

## Test de hipótesis sobre la espera media

El manual de operaciones afirma que la espera media en la línea de
inspección debería ser 11 minutos. Observamos $\bar{x} = 12$, $s = 3$,
$n = 36$.

$$ H_0: \mu = 11 \qquad H_1: \mu \neq 11 \qquad \alpha = 0{,}05 $$

**Paso 1.** Estadístico de prueba (mismo pivote (5.3)):

$$ T = \frac{\bar{X} - \mu_0}{s/\sqrt{n}} \tag{5.7} $$

**Paso 2.** Calculamos el $p$-valor en la $t_{n-1}$ y rechazamos si es
menor a $\alpha$.


In [ ]:
factory_test_input = OneSampleMeanTestInput(
    sample_mean=12.0,
    sample_standard_deviation=3.0,
    sample_size=36,
    null_mean=11.0,
    alternative=Alternative.TWO_SIDED,
    significance_level=0.05,
)
factory_test_result = test_one_sample_mean(factory_test_input)
factory_test_result

### Conexión IC ↔ test

Un test bilateral al nivel $\alpha$ rechaza $H_0: \mu = \mu_0$ **sí y solo
sí** $\mu_0$ no cae dentro del IC al nivel $1 - \alpha$ construido sobre la
misma muestra (ecuaciones (5.2) y (5.3)). Es el mismo objeto visto desde
dos ángulos.


## Bootstrap sin supuestos paramétricos

Cuando no queremos asumir Normalidad ni conocer $\sigma$, podemos
**remuestrear**: tomar muchas muestras con reemplazo del conjunto observado
y mirar la distribución de las medias bootstrap. El IC sale de los
percentiles.


In [ ]:
rng_bootstrap = np.random.default_rng(settings.random_seed)
synthetic_sample = Observations.validate(pd.DataFrame({"value": rng_bootstrap.normal(loc=12.0, scale=3.0, size=36)}))
bootstrap_input = BootstrapInput(
    observations=synthetic_sample,
    replicates=3_000,
    settings=settings,
)
bootstrap_result = bootstrap_mean(bootstrap_input)

bootstrap_chart_input = BootstrapDistributionChartInput(
    bootstrap_result=bootstrap_result,
    title="Distribución bootstrap de la media",
    settings=settings,
)
chart_bootstrap_distribution(bootstrap_chart_input)

## Tamaño muestral para una precisión dada

**Para la media.** Despejando $n$ de la fórmula (5.2) con margen de error
$E$ y $z_{1 - \alpha/2}$:

$$ n \ge \left(\frac{z_{1 - \alpha/2}\,\sigma}{E}\right)^2 \tag{5.8} $$

Para la clínica con $\sigma = 3$, $E = 1$ y 95%: $n \ge (1{,}96 \cdot 3)^2 \approx 35$.


In [ ]:
clinic_sample_size_input = SampleSizeForMeanInput(
    population_standard_deviation=3.0,
    margin_of_error=1.0,
    confidence_level=0.95,
)
clinic_sample_size = sample_size_for_mean(clinic_sample_size_input)
clinic_sample_size

**Para una proporción.** Despejando $n$ de (5.4) con $\hat{p} = 0{,}5$
(peor caso, varianza máxima):

$$ n \ge \left(\frac{z_{1 - \alpha/2}}{E}\right)^2 \hat{p}(1 - \hat{p}) \tag{5.9} $$


In [ ]:
poll_sample_size_input = SampleSizeForProportionInput(
    estimated_proportion=0.5,
    margin_of_error=0.03,
    confidence_level=0.95,
)
poll_sample_size = sample_size_for_proportion(poll_sample_size_input)
poll_sample_size

## Ejercicio 1 — IC contiene un valor de control

Construí un IC al 95% para $\mu$ con $\bar{x} = 12$, $\sigma = 3$,
$n = 36$. Verificá que contenga el valor de control $\mu_0 = 11{,}5$.


In [ ]:
exercise_known_input = MeanKnownVarianceInput(
    sample_mean=12.0,
    population_standard_deviation=3.0,
    sample_size=36,
    confidence_level=0.95,
)
exercise_interval = build_confidence_interval_for_mean_known_variance(exercise_known_input)
interval_contains_input = IntervalContainsInput(
    lower_bound=exercise_interval.lower_bound,
    upper_bound=exercise_interval.upper_bound,
    target_value=11.5,
)
verify_interval_contains(interval_contains_input)

## Ejercicio 2 — Tamaño muestral para una proporción

La encuesta quiere margen de error de ±3% al 95% de confianza, usando
$\hat{p} = 0{,}5$ (peor caso). **Idea.** Aplicar (5.9):

$$ n \ge \left(\frac{1{,}96}{0{,}03}\right)^2 \cdot 0{,}25 \approx 1067 $$


In [ ]:
exercise_proportion_input = SampleSizeForProportionInput(
    estimated_proportion=0.5,
    margin_of_error=0.03,
    confidence_level=0.95,
)
expected_size = sample_size_for_proportion(exercise_proportion_input).required_sample_size

student_answer_size = 1067.0
verify_input = NumericAnswerInput(
    student_answer=student_answer_size,
    expected_answer=float(expected_size),
    absolute_tolerance=1.0,
)
verify_numeric_answer(verify_input)

## Síntesis y respuestas

1. **Rango plausible para $\mu$ (clínica).** Aplicando (5.2) o (5.3) sobre
   $\bar{x} = 12$, $n = 36$, el IC al 95% es aproximadamente
   $(11{,}02,\ 12{,}98)$. Si conocemos $\sigma$ usamos Normal; si no,
   usamos $t_{n-1}$.
2. **IC para la proporción (encuesta).** Por (5.4),
   $0{,}5 \pm 1{,}96\sqrt{0{,}25/400} \approx (0{,}451,\ 0{,}549)$.
3. **¿Rechazamos la afirmación de la fábrica?** El test (5.7) compara
   $\bar{x} = 12$ con $\mu_0 = 11$ y rechaza $H_0$ al 5%; equivalentemente,
   $\mu_0 = 11$ no cae en el IC del punto 1.
4. **¿Cuántas encuestas?** Por (5.9), $n \ge 1067$ personas para asegurar
   margen de error ±3% al 95% en peor caso.


## Cierre del recorrido

Las cinco unidades miraron las mismas situaciones desde ángulos cada vez más
abstractos. La sala de espera empezó como una muestra de minutos descrita por
un boxplot, terminó modelada como Exponencial, su promedio diario apareció
junto al teorema central del límite y la espera media tuvo, finalmente, un
intervalo de confianza. La encuesta arrancó como conteo de respuestas, se
convirtió en un ejemplo de Bayes, después en una Binomial, en una proporción
asintóticamente Normal y, en este capítulo, en un IC con un tamaño muestral
calculado a propósito. La línea de producción pasó de defectos por turno a una
Bin/Poisson, a una aproximación Normal, hasta caer en un test de hipótesis
sobre la espera media de la inspección.

Si necesitás repasar **símbolos** o **términos**, el [glosario](glossary.md)
tiene tablas con todo lo que apareció. Si querés revisar **cómo está
construido pedagógicamente el libro**, el apéndice de
[enfoque pedagógico](pedagogy.md) explica la progresión que recorrieron las
secciones.
